# Advanced 03 — Contract-driven CrewAI teams

CrewAI orchestrates bounded work; application policy owns trust, authority, budgets, recovery, and completion. This notebook is credential-free through Part 15 and imports the same `policy.py` exercised by pytest.

In [ ]:
from pathlib import Path
import sys
COURSE_DIR = Path.cwd()
if not (COURSE_DIR / 'policy.py').exists():
    COURSE_DIR = Path('curriculum/advanced/03-crewai-teams').resolve()
sys.path.insert(0, str(COURSE_DIR))
import lab, policy
print(lab.QUESTION)

## Part 1 — Northstar task contract

The same incident and required evidence are used for every architecture. Stable task IDs—not execution position—identify work.

In [ ]:
tasks = lab.build_tasks()
[(task.task_id, task.expected_artifact_type.value, task.dependencies) for task in tasks]

## Part 2 — Agents versus authority

Role, goal, backstory, and visible tools configure behavior. `CapabilityPolicy` independently controls tenant and capability admission.

In [ ]:
workers = lab.build_agents()
capability_policy = lab.build_capability_policy()
[(worker.agent_id, worker.role.value, capability_policy.grants.get(worker.agent_id, ())) for worker in workers]

## Part 3 — Typed artifact envelopes

Pydantic enforces structure. Trust still requires producer, task, tenant, evidence, source, hash, policy, capability, and grounding checks.

In [ ]:
state = lab.build_flow_state()
health_artifact = lab.build_artifact('collect-health')
lab.accept_artifact(state, 'collect-health', health_artifact)
lab.artifact_table(state.accepted_artifacts.values())

## Part 4 — Task dependency validation

Admission rejects missing dependencies, cycles, self-dependencies, and incompatible input artifact types before any crew call.

In [ ]:
topological_order = policy.validate_task_graph(tasks)
topological_order

## Part 5 — Sequential execution

Sequential is a strong baseline for known stages. Application task state, rather than `kickoff()` returning, defines success.

In [ ]:
sequential = lab.run_same_workload('DETERMINISTIC_SEQUENTIAL')
sequential.model_dump()

## Part 6 — Parallel evidence tasks

The five evidence tasks form one ready set. Total work is their sum; conceptual batch wall time is their maximum.

In [ ]:
fresh = lab.build_flow_state()
ready = policy.ready_task_ids(tasks, fresh.task_states)
ready, sum(lab._DURATIONS[item] for item in ready), max(lab._DURATIONS[item] for item in ready)

## Part 7 — Artifact validation failures

Wrong-tenant and schema-valid unsupported outputs fail closed. Globex appears only as an adversarial case.

In [ ]:
failures = []
for candidate in [
    lab.build_artifact('collect-health', tenant_id='globex'),
    lab.build_artifact('collect-health', payload={'claims': [{'claim_id': 'x', 'text': 'database lost', 'evidence_ids': ['health'], 'fact_keys': ['database_loss']}]})
]:
    try:
        lab.accept_artifact(lab.build_flow_state(), 'collect-health', candidate)
    except policy.PolicyError as error:
        failures.append(str(error))
failures

## Part 8 — Hierarchical manager

The manager proposes a typed fallback task. The application validates the same worker, artifact-type, capability, tenant, and budget policy used elsewhere.

In [ ]:
manager_state = lab.build_flow_state()
decision = lab.build_recovery_decision()
policy.validate_manager_decision(decision, state=manager_state, crew=lab.build_crews()[2], capability_policy=lab.build_capability_policy(), known_task_ids=tuple(task.task_id for task in tasks))

## Part 9 — Manager budgets and failure modes

Unknown workers, fabricated parent lineage, worker capability mismatches, arbitrary task types, production writes, excessive depth/delegation, and repeated no-progress signatures are denied. `manager_calls` counts attempted model calls; `delegations` counts only accepted proposals.

In [ ]:
loop_state = lab.build_flow_state()
known_task_ids = tuple(task.task_id for task in tasks)
policy.validate_manager_decision(decision, state=loop_state, crew=lab.build_crews()[2], capability_policy=lab.build_capability_policy(), known_task_ids=known_task_ids)
try:
    policy.validate_manager_decision(decision, state=loop_state, crew=lab.build_crews()[2], capability_policy=lab.build_capability_policy(), known_task_ids=known_task_ids)
except policy.PolicyError as error:
    print(error)

## Part 10 — Recovery scenario

The primary deployment source fails. A bounded hierarchy earns its overhead only because the approved metadata fallback improves measured recovery.

In [ ]:
{name: metrics.model_dump() for name, metrics in lab.compare_recovery().items()}, lab.recovery_gate()

## Part 11 — CrewAI Flow control plane

The deterministic Flow chooses InvestigationCrew, validates artifacts, then chooses ReviewCrew. It owns global state, cancellation, and completion.

In [ ]:
completed_state, flow_metrics = lab.run_flow_controlled()
completed_state.terminal_status, len(completed_state.accepted_artifacts), flow_metrics.model_dump()

## Part 12 — Prompt injection and tenant tests

Retrieved text cannot emit a trusted Flow event. Tenant mismatch is rejected even when the artifact is structurally valid.

In [ ]:
lab.prompt_injection_is_inert(), {record.tenant_id for record in lab.build_evidence_registry().values()}

## Part 13 — Same-workload evaluation

Quality, grounding, recovery, calls, total work, wall clock, cost, duplicate rate, and privileged exposure are compared on one workload. A task record's `elapsed_ms` is individual work; only the scheduler computes batch wall-clock latency. The fixture's architecture cost ceiling is configuration, not a universal `$0.10` rule.

In [ ]:
import pandas as pd
pd.DataFrame([row.model_dump() for row in lab.compare_architectures()]).set_index('architecture')

## Part 14 — Sequential versus hierarchical versus Flow

The healthy case keeps sequential; the source-failure case accepts bounded hierarchy; known routing favors deterministic Flow over an irrelevant manager loop.

In [ ]:
{
    'healthy_case': lab.no_benefit_gate(),
    'recovery_case': lab.recovery_gate(),
    'bad_hierarchy_completed': lab.bad_hierarchy_metrics().completed,
    'context_projection': lab.context_projection_experiment(),
}

## Part 15 — Optional real CrewAI adapter

CrewAI 1.15.20 is the tested adapter version, not an architectural dependency. Offline replay checks SDK construction, Flow construction, and structured-output plumbing—not manager intelligence, delegation quality, or real-model reliability—and does not contribute to architecture-quality metrics. The direct `Task.context` chain below is a framework API demonstration only. The governed path runs a bounded task, parses its structured output, admits it through `admit_crewai_output()`, and passes only accepted artifacts downstream.

In [ ]:
import crewai_adapter
status = crewai_adapter.adapter_status()
print(status)
adapter_state = lab.build_flow_state()
candidate_payload = lab.build_artifact('collect-health').payload
accepted = crewai_adapter.admit_crewai_output(candidate_payload, task_id='collect-health', state=adapter_state)
print(accepted.artifact_id, adapter_state.task_states['collect-health'])
if status['installed']:
    offline_crew = crewai_adapter.build_sequential_crew()
    bounded_crew = crewai_adapter.build_bounded_task_crew('collect-logs', adapter_state)
    print(offline_crew.process, len(offline_crew.tasks), bounded_crew.tasks[-1].output_pydantic.__name__)

## Part 16 — Optional OpenAI-backed run

This paid path is opt-in via an existing `OPENAI_API_KEY`. It runs bounded task crews, parses each structured result, calls the same `admit_crewai_output()` policy boundary shown above, and stops on rejection before downstream work. No production write is available.

In [ ]:
import os
if os.getenv('OPENAI_API_KEY') and crewai_adapter.adapter_status()['installed']:
    print('Live run is configured. Call crewai_adapter.run_live_sequential_if_configured() only when you intend to spend API credits.')
else:
    print('Credential-free course complete; optional live run skipped.')

## Takeaway

Agent role is not authority. Typed output is not automatically trusted. `REVIEW_PASS` is not production approval. Hierarchy is justified only by measured benefit after safety, grounding, cost, and latency gates.